In [ ]:
# Check working directory

import os
print(os.getcwd())

In [ ]:
# I have set up the vector embedding using chromaDB and sentence transformers in the previous script. Now I will set up the RAG pipeline

# First, we reconnect to ChromaDB

import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

# Reconnect to existing ChromaDB collection
chroma_client = chromadb.PersistentClient(
    path="../data/snomed/chroma_db"
)

collection = chroma_client.get_collection(
    name="snomed_concepts"
)

print(f"Collection loaded: {collection.count()} concepts")

# Load embedding model — must be same model used during indexing
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

# Load OpenAI client
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client ready")

In [ ]:
# The retrieval function

def retrieve_snomed_concepts(query, n_results=400):
    """
    Given a clinical query string, retrieve the most semantically
    similar SNOMED-CT concepts from the vector store.
    
    Returns a list of dicts with 'code' and 'term' keys.
    """
    # Embed the query using the same model used during indexing
    query_embedding = model.encode([query])
    
    # Search ChromaDB for nearest neighbours
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    
    # Format results as a list of dicts
    retrieved = []
    for code, term in zip(results['ids'][0], results['documents'][0]):
        retrieved.append({'code': code, 'term': term})
    
    return retrieved

# Test the retrieval function
retrieved_concepts = retrieve_snomed_concepts("Type 2 Diabetes Mellitus", n_results=400)
print(f"Retrieved {len(retrieved_concepts)} concepts")
print("\nFirst 10 retrieved:")
for i, concept in enumerate(retrieved_concepts[:10]):
    print(f"{i+1}. [{concept['code']}] {concept['term']}")



In [ ]:
def build_rag_prompt(query, retrieved_concepts):
    """
    Build an augmented prompt using retrieved SNOMED-CT concepts
    as context for the LLM.
    """
    # Format retrieved concepts as a readable context block
    context = "\n".join([
        f"- {c['code']}: {c['term']}" 
        for c in retrieved_concepts
    ])
    
    messages = [
        {
            "role": "system",
            "content": "You are a clinical terminology expert supporting EHR research in UK primary care."
        },
        {
            "role": "user",
            "content": f"""You are building a SNOMED-CT codelist for defining a research cohort.

The following SNOMED-CT concepts have been retrieved from the ontology as potentially relevant to: {query}

RETRIEVED CONCEPTS:
{context}

Using these retrieved concepts as a reference, generate a complete SNOMED-CT codelist for {query}.

Requirements:
- Select only the most clinically relevant codes from the retrieved list above
- You may supplement with additional valid SNOMED-CT codes from your own knowledge where clinically justified
- Do not include any codes that are not clinically relevant to {query}
- Codes must be valid SNOMED-CT concept identifiers
- Return JSON only, no preamble, no explanation, no markdown formatting

Return in this exact format:
[
  {{"code": "SNOMED_CODE", "term": "DESCRIPTION"}}
]"""
        }
    ]
    
    return messages

# Test it
messages = build_rag_prompt(
    "Type 2 Diabetes Mellitus", 
    retrieved_concepts
)

# Print the user message to inspect it
print(messages[1]['content'][:500])
print("...")

In [ ]:
# Make the RAG API call
response_rag = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    temperature=0,
    max_completion_tokens=5000,
)

# Extract and parse the response
raw_content_rag = response_rag.choices[0].message.content

# Strip markdown fences if present
cleaned_rag = raw_content_rag.strip()
if cleaned_rag.startswith("```"):
    cleaned_rag = "\n".join(cleaned_rag.split("\n")[1:-1])

# Parse JSON
llm_codes_rag = json.loads(cleaned_rag)

# Convert to dataframe
llm_df_rag = pd.DataFrame(llm_codes_rag)
print(f"RAG codelist generated: {len(llm_df_rag)} codes")
print(llm_df_rag.head(10).to_string())

In [ ]:
print(f"Response length: {len(raw_content_rag)} characters")
print(f"Last 300 characters:")
print(raw_content_rag[-300:])

In [ ]:
print(raw_content_rag[-200:])

In [ ]:
import pandas as pd

# Load gold standard
gold_standard = pd.read_csv(
    "../data/gold_standards/nhsd-primary-care-domain-refsets-dmtype2_cod-20250912.csv",
    dtype={'code': str}
)

# Convert to sets
gold_codes = set(gold_standard['code'])
rag_codes_set = set(llm_df_rag['code'])

# Calculate components
true_positives_rag = gold_codes & rag_codes_set
false_positives_rag = rag_codes_set - gold_codes
false_negatives_rag = gold_codes - rag_codes_set

# Calculate metrics
precision_rag = len(true_positives_rag) / len(rag_codes_set)
recall_rag = len(true_positives_rag) / len(gold_codes)
f1_rag = 2 * (precision_rag * recall_rag) / (precision_rag + recall_rag)

print("=== RAG RESULTS ===")
print(f"True Positives:  {len(true_positives_rag)}")
print(f"False Positives: {len(false_positives_rag)}")
print(f"False Negatives: {len(false_negatives_rag)}")
print(f"\nPrecision: {precision_rag:.3f}")
print(f"Recall:    {recall_rag:.3f}")
print(f"F1:        {f1_rag:.3f}")

print("\n=== COMPARISON ===")
print(f"{'Metric':<12} {'Non-RAG':>10} {'RAG':>10} {'Change':>10}")
print(f"{'Precision':<12} {0.136:>10.3f} {precision_rag:>10.3f} {precision_rag - 0.136:>+10.3f}")
print(f"{'Recall':<12} {0.130:>10.3f} {recall_rag:>10.3f} {recall_rag - 0.130:>+10.3f}")
print(f"{'F1':<12} {0.133:>10.3f} {f1_rag:>10.3f} {f1_rag - 0.133:>+10.3f}")

In [ ]:
# What did RAG get right?
print("=== TRUE POSITIVES ===")
correct_rag = gold_standard[gold_standard['code'].isin(true_positives_rag)]
print(correct_rag.to_string())

# What did RAG hallucinate or add beyond gold standard?
print("\n=== FALSE POSITIVES ===")
hallucinated_rag = llm_df_rag[llm_df_rag['code'].isin(false_positives_rag)]
print(hallucinated_rag.to_string())

# What did RAG miss?
print("\n=== FALSE NEGATIVES ===")
missed_rag = gold_standard[gold_standard['code'].isin(false_negatives_rag)]
print(missed_rag.to_string())

# Critical check: were the missed codes even retrieved?
print("\n=== RETRIEVAL COVERAGE CHECK ===")
retrieved_codes = set([c['code'] for c in retrieved_concepts])
missed_but_retrieved = false_negatives_rag & retrieved_codes
missed_not_retrieved = false_negatives_rag - retrieved_codes
print(f"Missed codes that WERE retrieved: {len(missed_but_retrieved)}")
print(f"Missed codes that were NOT retrieved: {len(missed_not_retrieved)}")
print("\nCodes missed because not retrieved:")
for code in missed_not_retrieved:
    term = gold_standard[gold_standard['code']==code]['term'].values
    print(f"  {code}: {term[0] if len(term) > 0 else 'unknown'}")

In [ ]:
help(chromadb)

In [ ]:
# Trying some edits to the RAG pipeline

In [ ]:
# Import libraries

import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
print("Imports done")

In [ ]:
# load chroma

chroma_client = chromadb.PersistentClient(
    path="../data/snomed/chroma_db"
)
print("Client initialised")

In [ ]:
# Chroma collection

collection = chroma_client.get_collection(name="snomed_concepts")
print(f"Collection loaded: {collection.count()} concepts")

In [ ]:
# Embedding model

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

In [ ]:
print("Hello World")